# Phase 4 Scale-Up: OpenWebText Fock-PARFLM v2.1 with **Structured V_θ**

## Motivation

The structured-V_θ TinyStories sweep showed that the **Multi-Xi Fock-PARFLM
v2.1** reaches **10.36 PPL** with a drop-in **SQ3 mixture-of-quadratic-wells
V_θ** (`K_mix=8`, `K_xi=4`, arm A2) — within ~1.06 PPL of the MLP baseline
(8.95 PPL) while replacing the MLP V_θ with an analytically-differentiable,
attractor-explicit parameterisation. This Phase 4 notebook scales that **A2
winner config** from TinyStories to **OpenWebText**, to establish the
structured-V_θ PPL floor for the Fock architecture at scale.

**This run is a 200k-step diagnostic** — same architecture as the intended
1M-step run, so the final PPL is a reliable early-stage estimate. The run can
be extended to 1M steps later by simply changing `TOTAL_STEPS`.

**Why structured V_θ at scale matters:** the SQ3 V_θ uses analytical gradients
(no `create_graph` cost on V_θ) and exposes its `K_mix` attractor centres in
closed form (interpretability with no gradient-descent extraction). Confirming
that the small expressivity gap observed on TinyStories holds on OpenWebText is
the key open question from the structured-V_θ companion note.

**Scaling knobs (and why `d` is kept near the A2 value).** The SQ3 projections
grow as `K_ξ · K_mix · d²`, so pushing `d` to 512 would inflate V_θ past the
MLP it replaces and partly defeats the structured-V_θ rationale. The faithful,
memory-cheap scale-up levers are therefore **depth `L`** (more conservative
Verlet layers — nearly free under per-layer checkpointing), the **Fock
register pool `M`**, and above all the **corpus (40×) and step budget
(12.5×)**. `d` is bumped only modestly (256 → 384) at the top tier.

## Config summary

| Parameter | TinyStories A2 | **Phase 4 OWT (this notebook)** |
|-----------|----------------|--------------------------------|
| `d` | 256 | **384** (adaptive fallback: 256) |
| `L` | 8 | **16** (adaptive fallback: 12 / 8) |
| V_θ | SQ3 `K_mix=8` | SQ3 `K_mix=8` (unchanged) |
| `xi_channels` (K_ξ) | 4 | 4 |
| `n_registers` (Fock) | 16 | **32** (fallback: 16) |
| `top_k` (V_φ) | 8 | 8 |
| `fixed_gamma` | 0.30 | 0.30 |
| `lambda_V` | 1e-2 | 1e-2 |
| Total steps | 16,000 | **200,000** (extendable to 1M) |
| Checkpoints | every 4k | **every 25k (8 saves)** |
| Corpus | TinyStories (~5M tok) | **OpenWebText (~200M tok)** |
| Target GPU | A100/T4 | **H100 (80 GB)** |

## H100 memory strategy

The Fock-PARFLM v2.1 + structured-V_θ stack is kept comfortably within an
H100's 80 GB by:
- **Gathered top-k V_φ** (`use_gathered_v_phi=True`) — only the `top_k`
  neighbours enter the pairwise potential graph instead of the full `T×T`.
- **Per-layer activation checkpointing** (`use_layer_checkpoint=True`) — one
  layer's Verlet graph is held at a time.
- **Structured V_θ** — analytical gradients keep the V_θ graph tiny.
- An **adaptive sizing probe** that tries `d=384, L=16, M=32` first and falls
  back to smaller tiers on OOM, then an **auto batch-size** probe.

## Stability (fix after the first OWT blowup)

The first OWT attempt trained cleanly to ~307 PPL by 16k steps, then the
`V_θ²` regulariser ran away (`v_reg → 2e5`, so `λ_V·v_reg ≫ NTP`), producing
1e8 gradients and a permanent divergence the clip could not undo. The
quadratic SQ3 potential is unbounded in `h`, so `mean(V_θ²)` is outlier-driven.
Three rounds of fixes have been applied:

**Round 1** (after first blowup at step 5k, v_reg→2e5): bounded `log1p(V_θ²)`
penalty; LR 3e-4→2e-4; warmup 5k→8k; non-finite-update skip guard; rolling
best checkpoint (`*_best.pt`).

**Round 2** (after soft blowup at step ~22k, grad_norm→214, NTP 5.6→6.7):
- LR 2e-4 → **1.5e-4** (less headroom for large gradient directions)
- grad_clip 0.5 → **0.3** (tighter — 214-norm batch with clip=0.5 still applies
  a destabilizing direction; 0.3 forces a smaller step)
- **Cell 2 now auto-selects `*_best.pt`** when it is more recent than the latest
  periodic step checkpoint, so a blowup between two 25k saves is always
  recoverable without manual file renaming.
- **EMA gradient-norm watchdog**: if the 50-step EMA of the raw gradient norm
  stays above 50 for 100 consecutive steps (= soft-instability spiral), the
  watchdog automatically reloads `*_best.pt` in-place and continues training.

**Round 3** (persistent "doom loop" — 25+ watchdog triggers between steps 22k–33k
kept reloading the same best checkpoint, preventing progress):
- **Fix A — Force clamping** (`force_clamp_max=10.0`): per-dim clamp on the
  conservative force before the Verlet integrator.  Bounds the maximum
  displacement from any single force evaluation, eliminating the tail of extreme
  V_theta gradients.
- **Fix B — Separate V_theta learning rate** (`V_THETA_LR_MULT=0.1`): the
  V_theta / ln_before_v parameters get 0.1× the main LR, preventing the
  amplified SQ3 backward-graph gradients from destabilising well centres.
- **Fix C — LayerNorm before V_theta** (`ln_before_vtheta=True`): normalises
  `h` before V_theta evaluation so that `||h − μ_k||` stays bounded regardless
  of hidden-state magnitude growth across layers.
- **Watchdog recalibration**: threshold raised 3.5 → **10.0**, patience raised
  30 → **100** steps, to avoid false-positive triggers now that fixes A/B/C
  structurally limit force magnitudes.

## Multi-session strategy

This notebook is designed for **checkpoint-to-checkpoint** execution across
multiple H100 sessions. Each session:
1. Mounts Drive, finds the latest checkpoint
2. Restores model + optimizer state
3. Trains until the next checkpoint (or session timeout)
4. Saves checkpoint to Drive and exits cleanly

Checkpoints save every 25k steps (8 total), so at most ~25k steps of compute
is lost on an unexpected disconnect.

In [ ]:
# ── Cell 1: Environment + Drive mount ─────────────────────────────
import subprocess, sys, os, gc, math, json, time
from pathlib import Path
from dataclasses import asdict

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    run('pip install -q transformers huggingface_hub pyarrow datasets')
    if not os.path.isdir('semsimula-paper'):
        run('git clone --depth 1 https://github.com/dimitarpg13/semsimula-paper.git')
    REPO = 'semsimula-paper'
else:
    REPO = os.environ.get('SEMSIMULA_PAPER', '.')

ARCH_DIR = os.path.join(REPO, 'notebooks', 'conservative_arch')
for p in [
    ARCH_DIR,
    os.path.join(ARCH_DIR, 'multixi'),
    os.path.join(ARCH_DIR, 'parf'),
    os.path.join(ARCH_DIR, 'sarf_mass_variant'),
    os.path.join(ARCH_DIR, 'energetic_minima'),
    os.path.join(ARCH_DIR, 'scaleup'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
if torch.cuda.is_available():
    # Allow TF32 on H100 for the dense matmuls (V_phi, embeddings); the
    # conservative dynamics are robust to it at this scale.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else ('mps' if hasattr(torch.backends, 'mps')
          and torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    GPU_NAME = torch.cuda.get_device_name()
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_structured_vtheta_openwebtext_phase4')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_fock_structured_vtheta_openwebtext_phase4'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = DRIVE_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive root   : {DRIVE_ROOT}')
print(f'Checkpoints  : {CKPT_DIR}')
print(f'Results      : {RESULTS_DIR}')

In [ ]:
# ── Cell 2: Checkpoint resolution + resume detection ─────────────
TOTAL_STEPS   = 200_000          # diagnostic; bump to 1_000_000 later
CKPT_INTERVAL = 25_000           # 8 saves over 200k steps
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))
CKPT_PREFIX   = 'fock_structured_vtheta_owt_phase4'

resume_step = 0
resume_ckpt = None

# ── 1. Scan periodic step checkpoints ─────────────────────────────
for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

# ── 2. Also consider *_best.pt as a resume candidate ──────────────
# After a soft blowup, the periodic step checkpoint may contain diverged
# weights while *_best.pt still holds the pre-blowup best model.  We
# prefer *_best.pt whenever it is MORE RECENT than the latest step
# checkpoint, so a blowup between two step saves is always recoverable.
_best_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _best_path.exists():
    try:
        import torch as _t
        _bd = _t.load(_best_path, map_location='cpu', weights_only=False)
        _best_step = _bd.get('step', 0)
        _best_ppl  = _bd.get('val_ppl', float('inf'))
        del _bd
        if _best_step > resume_step:
            # best.pt is more recent → use it (step ckpt is older / absent)
            print(f'*_best.pt (step {_best_step:,}, PPL {_best_ppl:.2f}) is more recent '
                  f'than latest step checkpoint (step {resume_step:,}) — resuming from best.')
            resume_ckpt = _best_path
            resume_step = _best_step
        elif _best_step == resume_step:
            print(f'Both step checkpoint and *_best.pt are at step {resume_step:,}; '
                  f'using step checkpoint.')
        else:
            print(f'Step checkpoint (step {resume_step:,}) is more recent than '
                  f'*_best.pt (step {_best_step:,}); using step checkpoint.')
    except Exception as e:
        print(f'[warn] could not inspect *_best.pt: {e}')

if resume_ckpt is not None:
    print(f'\nResuming from: {resume_ckpt.name}  (step {resume_step:,})')
    print(f'Remaining: {TOTAL_STEPS - resume_step:,} steps')
else:
    print('No checkpoint found — training from scratch.')
    print(f'Total: {TOTAL_STEPS:,} steps  Checkpoints every {CKPT_INTERVAL:,}')

In [ ]:
# ── Cell 3 (optional): Progress review — run at session start ─────
# Safe to run WITHOUT loading model or data. Just reads Drive files.
# Shows best PPL so far, checkpoint history, and a quick loss curve.
import json, math
from pathlib import Path

_ckpt_dir    = DRIVE_ROOT / 'checkpoints'
_results_dir = DRIVE_ROOT / 'results'
_log_path    = _results_dir / 'training_log.jsonl'
_prefix      = CKPT_PREFIX
_total       = TOTAL_STEPS

print('─' * 55)
print('PHASE 4 PROGRESS REPORT — Fock-PARFLM v2.1 + structured V_theta')
print('─' * 55)

# ── 1. Checkpoint summary ──────────────────────────────────────────
print('\nCheckpoints on Drive:')
ckpt_records = []
for ckpt_path in sorted(_ckpt_dir.glob(f'{_prefix}_step*.pt')):
    try:
        d = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        step = d.get('step', 0)
        ppl  = d.get('val_ppl', math.exp(d.get('val_loss', float('nan'))))
        ckpt_records.append((step, ppl, ckpt_path.name))
    except Exception as e:
        print(f'  [warn] could not read {ckpt_path.name}: {e}')

if ckpt_records:
    ckpt_records.sort()
    for step, ppl, name in ckpt_records:
        bar = '█' * int(step / _total * 30)
        print(f'  step {step:>7,}  PPL {ppl:>8.2f}  {bar}')
    best_step, best_ppl, best_name = min(ckpt_records, key=lambda x: x[1])
    print(f'\n  Best PPL so far : {best_ppl:.2f}  at step {best_step:,}  ({best_name})')
    last_step = ckpt_records[-1][0]
    print(f'  Progress        : {last_step:,} / {_total:,} steps  '
          f'({100*last_step/_total:.1f}%)')
else:
    print('  No checkpoints yet.')

# ── 2. Training log summary ────────────────────────────────────────
print('\nVal PPL from training_log.jsonl (all sessions):')
eval_entries = []
if _log_path.exists():
    with open(_log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e:
                    eval_entries.append((e['step'], e['val_ppl']))
            except Exception:
                pass

if eval_entries:
    eval_entries.sort()
    best_log_ppl = min(p for _, p in eval_entries)
    last_log_step, last_log_ppl = eval_entries[-1]
    print(f'  Total eval entries : {len(eval_entries)}')
    print(f'  Last eval          : step {last_log_step:,}  PPL {last_log_ppl:.2f}')
    print(f'  Best eval PPL      : {best_log_ppl:.2f}')

    ppls = [p for _, p in eval_entries]
    mn, mx = min(ppls), max(ppls)
    if mx > mn:
        blocks = ' ▁▂▃▄▅▆▇█'
        spark = ''.join(blocks[min(8, int((p - mn) / (mx - mn) * 8))] for p in ppls[-60:])
        print(f'\n  PPL trend (last {min(60,len(ppls))} evals, low=▁ high=█):')
        print(f'  {spark}')
else:
    print('  No eval entries yet.')

print('\n' + '─' * 55)

In [ ]:
# ── Cell 4: Data loading (reuse cached OpenWebText from earlier phases) ──
from data_module import get_batch

MAX_TRAIN_TOKENS = 200_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000
VOCAB_SIZE       = 50257

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

# Search all prior phase data caches (reuse the same tokenised OWT shards)
for alt_name in [
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        import shutil
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens)...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

# ── Logfreq surprisal (logfreq mass mode) ──
LOGFREQ_PATH = str(DATA_DIR / 'logfreq_surprisal_openwebtext.npy')
for alt_name in [
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
]:
    if os.path.exists(LOGFREQ_PATH):
        break
    if IN_COLAB:
        alt_lf = Path(f'/content/drive/MyDrive/{alt_name}/data/logfreq_surprisal_openwebtext.npy')
    else:
        alt_lf = Path.home() / alt_name / 'data' / 'logfreq_surprisal_openwebtext.npy'
    if alt_lf.exists():
        import shutil
        print(f'Reusing logfreq from {alt_lf}')
        shutil.copy2(str(alt_lf), LOGFREQ_PATH)
        break

if not os.path.exists(LOGFREQ_PATH):
    print('Computing unigram surprisal from OpenWebText training tokens...')
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.int64)
    N = len(train_ids)
    p = (counts + 1.0) / (N + VOCAB_SIZE)
    surprisal = -np.log(p).astype(np.float32)
    print(f'  Surprisal: min={surprisal.min():.3f}  max={surprisal.max():.3f}  '
          f'mean={surprisal.mean():.3f}')
    np.save(LOGFREQ_PATH, surprisal)
    del counts, p, surprisal

gc.collect()
print(f'\nReady: train={len(train_ids):,}  val={len(val_ids):,}')
print(f'Logfreq: {LOGFREQ_PATH}')

In [ ]:
# ── Cell 5: Model config + structured V_theta swap (adaptive sizing) ──
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2          # noqa: F401  (registers Fock v2 layer step)
import model_parf_multixi          # noqa: F401
import model_parf                  # noqa: F401
import model_parf_sparse           # noqa: F401
from model_structured_vtheta import MixtureQuadraticVTheta
from model_structured_vtheta_multixi import StructuredVThetaMultiXiAdapter

# ── Structured V_theta recipe = TinyStories A2 winner (unchanged) ──
V_THETA_KIND   = 'sq3'             # SQ3 mixture-of-quadratic-wells
K_MIX          = 8                 # SPLM/Fock winner mixture count
TAU            = 1.0
LAMBDA_V       = 1e-2              # V_theta^2 regulariser weight
XI_CHANNELS    = 4
XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]

BLOCK_SIZE = 512

# Adaptive architecture tiers (d, L, n_registers): try most aggressive first.
# V_theta stays SQ3(K_mix=8).  We scale DEPTH (L) and the Fock register pool
# (M) — both nearly free under per-layer checkpointing — plus a modest width
# bump (d=256 -> 384).  d is kept near the A2 value on purpose: the SQ3
# projections grow as K_xi * K_mix * d^2, so a large d would inflate V_theta
# past the MLP it replaces.  The real scale-up is the corpus (40x) + steps.
ARCH_TIERS = [
    (384, 16, 32),   # target:     d=384  L=16  M=32
    (384, 12, 16),   # fallback 1: shallower + smaller pool
    (256, 16, 16),   # fallback 2: A2 width, 2x depth
    (256,  8, 16),   # fallback 3: TinyStories-proven config (guaranteed fit)
]


def make_config(d, L, n_registers):
    return FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L, v_hidden=1024, v_depth=3, dt=1.0,
        mass_mode='logfreq',
        logfreq_path=LOGFREQ_PATH,
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=0.30,
        causal_force=True,
        ln_after_step=True,
        # Stability (Fixes A+C from Training Instabilities note)
        force_clamp_max=10.0,          # Fix A: clamp per-dim force to [-10, 10]
        ln_before_vtheta=True,         # Fix C: LN(h) before V_theta eval
        # Multi-Xi
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        # PARFLM (sparse, gathered, checkpointed)
        v_phi_kind='structural_competitive',
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        top_k=8,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        # Fock v2.1
        fock_version='v2',
        n_registers=n_registers,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=64,
        stack_discipline=True,
        d_k=64,
        tau_create_init=8.0,
        reverse_channel=True,
        per_register_tau=True,
        per_register_keys=True,
        ortho_register_init=True,
    )


def build_sq3_vtheta(xi_d, d, K_mix, tau, device):
    """Build an SQ3 MixtureQuadraticVTheta with asymmetric dims:
    input = flattened multi-xi context (xi_d = K_xi * d), output wells in
    the model hidden space (d).  Wrapped in the Multi-Xi adapter.
    """
    inner = MixtureQuadraticVTheta.__new__(MixtureQuadraticVTheta)
    nn.Module.__init__(inner)
    inner.d = d
    inner.K = K_mix
    inner.tau = tau
    inner.mu_proj = nn.Linear(xi_d, K_mix * d)
    inner.a_proj  = nn.Linear(xi_d, K_mix * d)
    inner.pi_proj = nn.Linear(xi_d, K_mix)
    inner.b_proj  = nn.Linear(xi_d, 1)
    inner._init_weights(0.0)
    return StructuredVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=d).to(device)


def try_model(d, L, n_registers):
    cfg = make_config(d, L, n_registers)
    mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
    n_v_theta_mlp = sum(p.numel() for p in mdl.V_theta.parameters())
    # Swap MLP V_theta -> structured SQ3 (analytical-grad, ~100x smaller)
    xi_d = XI_CHANNELS * d
    mdl.V_theta = build_sq3_vtheta(xi_d, d, K_MIX, TAU, DEVICE)
    n = mdl.num_params()
    n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
    print(f'  Trying d={d} L={L} M={n_registers} -> {n:,} params  '
          f'(V_theta {n_v_theta_mlp:,} MLP -> {n_v_theta:,} SQ3)')
    if DEVICE == 'cuda':
        _rng = np.random.default_rng(42)
        _xb, _yb = get_batch(train_ids, 2, BLOCK_SIZE, _rng)
        _x = torch.from_numpy(_xb).to(DEVICE)
        _y = torch.from_numpy(_yb).to(DEVICE)
        _, _loss = mdl(_x, _y)
        _loss.backward()
        mdl.zero_grad(set_to_none=True)
        del _x, _y, _xb, _yb, _loss
        torch.cuda.empty_cache()
        print(f'  OOM probe passed (batch=2)')
    return mdl, cfg


model = None
model_cfg = None
for d_try, L_try, M_try in ARCH_TIERS:
    try:
        model, model_cfg = try_model(d_try, L_try, M_try)
        break
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d_try} L={L_try} M={M_try} — trying next tier...')
            model = None
            torch.cuda.empty_cache()
            gc.collect()
        else:
            raise

if model is None:
    raise RuntimeError('Could not fit model at any tier')

# ── Auto batch size ──
GRAD_ACCUM = 1
if DEVICE == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram >= 70:
        BATCH_SIZE = 8
    elif vram >= 40:
        BATCH_SIZE = 4
        GRAD_ACCUM = 2
    else:
        BATCH_SIZE = 2
        GRAD_ACCUM = 4
    try:
        _rng = np.random.default_rng(42)
        _xb, _yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, _rng)
        _x = torch.from_numpy(_xb).to(DEVICE)
        _y = torch.from_numpy(_yb).to(DEVICE)
        _, _loss = model(_x, _y)
        _loss.backward()
        model.zero_grad(set_to_none=True)
        del _x, _y, _xb, _yb, _loss
        torch.cuda.empty_cache()
        print(f'OOM probe passed at batch_size={BATCH_SIZE}')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        model.zero_grad(set_to_none=True)
        gc.collect()
        old_bs = BATCH_SIZE
        BATCH_SIZE = max(BATCH_SIZE // 2, 1)
        GRAD_ACCUM = max(old_bs * GRAD_ACCUM // BATCH_SIZE, 2)
        print(f'OOM at batch_size={old_bs} — falling back to '
              f'batch_size={BATCH_SIZE} x {GRAD_ACCUM} grad accum')
else:
    BATCH_SIZE = 2
    GRAD_ACCUM = 2

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
alpha_init_str = ','.join(f'{a:.3f}' for a in model.xi_alpha_values())

print(f'\nModel: FockMultiXiPARFLM v2.1 + structured V_theta (Phase 4)')
print(f'  params: {n_params:,}  (V_theta SQ3: {n_v_theta:,})')
print(f'  d={model_cfg.d}  L={model_cfg.L}  n_registers={model_cfg.n_registers}')
print(f'  V_theta=SQ3(K_mix={K_MIX}, tau={TAU})  lambda_V={LAMBDA_V}')
print(f'  xi_channels={model_cfg.xi_channels}  alpha_init=[{alpha_init_str}]')
print(f'  v_phi=structural_competitive  top_k={model_cfg.top_k}  fixed_gamma={model_cfg.fixed_gamma}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')

In [ ]:
# ── Cell 6: Training loop (200k steps, checkpoint every 25k) ──────
LR            = 1.2e-4    # reduced 1.5e-4→1.2e-4: blowup at step ~23k with 1.5e-4
V_THETA_LR_MULT = 0.1     # Fix B: V_theta gets 0.1x LR (deep backward graph amplifies its gradients)
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = 8000      # gentler ramp (unchanged)
GRAD_CLIP     = 0.25      # tightened 0.3→0.25: Fock+SQ3 backward graph needs tighter clip
EVAL_INTERVAL = 2000
EVAL_ITERS    = 40
LOG_INTERVAL  = 200
SEED          = 0

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(x, targets, lambda_v):
    """Next-token loss + lambda_v * mean(V_theta^2).

    Recomputes the stack once; the structured V_theta is evaluated on the
    final hidden states to form the V_theta^2 regulariser that keeps the
    learned scalar potential compact (matches the TinyStories A2 recipe).
    """
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = h_L @ model.E.weight.T
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        h_for_v = model.ln_before_v(h_L) if model.ln_before_v is not None else h_L
        V_vals = model.V_theta(xis, h_for_v)
        # BOUNDED penalty (log1p) instead of the plain mean(V^2).
        # The SQ3 mixture potential is quadratic in h, so when a token drifts
        # far from every well V_theta blows up and mean(V^2) becomes an
        # outlier-dominated, unbounded loss term — the OWT run hit v_reg ~ 2e5,
        # so lambda_v * v_reg ~ 2000 dwarfed the NTP loss (~6), produced 1e8
        # gradients, and the model never escaped the penalty-minimising runaway.
        # log1p(V^2) matches V^2 for V ~ O(1) (same landscape compression in the
        # normal regime) but its loss grows only logarithmically and its
        # gradient is bounded (|2V/(1+V^2)| <= 1), so it can never dominate.
        v_reg_value = torch.log1p(V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp
    return loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():               # Fock dynamics need grad
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'lambda_v': LAMBDA_V, 'v_theta_lr_mult': V_THETA_LR_MULT,
            'force_clamp_max': 10.0, 'ln_before_vtheta': True,
        },
        'v_theta_recipe': {
            'kind': V_THETA_KIND, 'K_mix': K_MIX, 'tau': TAU,
            'xi_channels': XI_CHANNELS,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'variant': 'fock_parf_multixi_v2.1_structured_vtheta',
        'experiment': (
            f'openwebtext_fock_structured_vtheta_phase4_'
            f'd{model_cfg.d}_L{model_cfg.L}_M{model_cfg.n_registers}_'
            f'sq3K{K_MIX}'
        ),
        'corpus': 'openwebtext',
        'phase': 4,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    torch.save(ckpt, path)
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    return path


# ── Optimizer (Fix B: separate V_theta param group with lower LR) ──
_v_theta_ids = {id(p) for p in model.V_theta.parameters()}
_ln_bv_ids = {id(p) for p in model.ln_before_v.parameters()} if model.ln_before_v is not None else set()
_vtheta_group_ids = _v_theta_ids | _ln_bv_ids
v_theta_params = [p for p in model.parameters() if p.requires_grad and id(p) in _vtheta_group_ids]
other_params   = [p for p in model.parameters() if p.requires_grad and id(p) not in _vtheta_group_ids]
print(f'  Optimizer: main group {sum(p.numel() for p in other_params):,} params @ LR={LR}')
print(f'             V_theta group {sum(p.numel() for p in v_theta_params):,} params @ LR={LR * V_THETA_LR_MULT}')
optim = torch.optim.AdamW([
    {'params': other_params,   'lr': LR},
    {'params': v_theta_params, 'lr': LR * V_THETA_LR_MULT},
], weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))

# ── Resume from checkpoint ──
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    _missing, _unexpected = model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    if _missing:
        print(f'  [info] Missing keys in checkpoint (new modules, init from scratch): {_missing}')
    if 'optimizer_state_dict' in ckpt_data:
        try:
            optim.load_state_dict(ckpt_data['optimizer_state_dict'])
            print(f'  Optimizer state restored.')
        except (ValueError, KeyError) as _e:
            print(f'  [info] Optimizer state incompatible (param-group change), starting fresh: {_e}')
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    print(f'  Continuing from step {resume_step + 1:,} to {TOTAL_STEPS:,}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ── Training log ──
log_path = RESULTS_DIR / 'training_log.jsonl'
log_f = log_path.open('a')

t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
n_run = 0
last_avg_ntp = float('nan')
n_skipped = 0

# Track the best model across all sessions so a late blowup can never
# destroy it (rolling interval checkpoints alone would be overwritten by
# diverged weights).  Restore the running best if one already exists.
best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL from previous session: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] could not read best checkpoint: {e}')

# ── EMA gradient-norm watchdog ────────────────────────────────────
# If the EMA of the raw gradient norm stays elevated for many consecutive
# steps, the model is in a soft-instability spiral that clipping alone
# cannot fix.  We detect this and reload the best checkpoint automatically,
# then continue training with the LR already at its cosine-scheduled value
# (no warmup repeat — the optimizer is also restored).
GRAD_NORM_EMA_ALPHA = 0.05          # ~20-step memory (unchanged)
GRAD_NORM_EMA_THRESHOLD = 10.0      # raised 3.5→10.0: force clamp + LN + split LR remove the
                                    # mild spikes that were false-positives at 3.5; only catch
                                    # genuine blowups where EMA climbs well above clamp-induced baseline
GRAD_NORM_EMA_PATIENCE = 100        # raised 30→100: with fixes A/B/C the model needs room to work
                                    # through hard OWT clusters without premature reload
_grad_norm_ema = 0.0
_grad_norm_above_thresh_count = 0

def _reload_best_checkpoint():
    """Reload *_best.pt into model + optim in-place. Returns the step stored."""
    if not _best_ckpt_path.exists():
        print('[watchdog] *_best.pt not found — cannot reload.')
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    if 'optimizer_state_dict' in ckpt:
        try:
            optim.load_state_dict(ckpt['optimizer_state_dict'])
        except (ValueError, KeyError):
            pass
    step_in_ckpt = ckpt.get('step', 0)
    ppl_in_ckpt  = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best checkpoint: step {step_in_ckpt:,}  PPL {ppl_in_ckpt:.2f}')
    return step_in_ckpt

ckpt_steps_set = set(CKPT_STEPS)
steps_this_session = 0

print(f'\n{"="*60}')
print(f'Phase 4 Scale-Up (Fock + structured V_theta): steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  warmup={WARMUP_STEPS}  grad_clip={GRAD_CLIP}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}  '
      f'V_theta=SQ3(K={K_MIX})  params={n_params:,}')
print(f'  checkpoints every {CKPT_INTERVAL:,} steps')
print(f'{"="*60}\n')

for step in range(resume_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    optim.param_groups[0]['lr'] = lr_now
    optim.param_groups[1]['lr'] = lr_now * V_THETA_LR_MULT

    optim.zero_grad(set_to_none=True)
    accum_ntp = 0.0
    accum_vreg = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        loss, loss_ntp, v_reg = forward_with_vreg(x, y, LAMBDA_V)
        (loss / GRAD_ACCUM).backward()
        accum_ntp  += loss_ntp.item()       / GRAD_ACCUM
        accum_vreg += float(v_reg.detach()) / GRAD_ACCUM

    grad_norm = nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    )
    # Insurance: never apply a non-finite update.
    if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
        optim.step()
    else:
        n_skipped += 1
        optim.zero_grad(set_to_none=True)
        print(f'  [skip] non-finite grad/loss at step {step+1} '
              f'(grad_norm={float(grad_norm):.3e}, ntp={accum_ntp:.3e}) — update skipped '
              f'(total skipped: {n_skipped})')

    # ── EMA watchdog: detect soft-instability spiral ───────────────
    _raw_gn = float(grad_norm)
    _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
    if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
        _grad_norm_above_thresh_count += 1
    else:
        _grad_norm_above_thresh_count = 0

    if _grad_norm_above_thresh_count >= GRAD_NORM_EMA_PATIENCE:
        print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
              f'for {_grad_norm_above_thresh_count} steps — soft blowup detected at step {step+1}.')
        _reload_best_checkpoint()
        _grad_norm_ema = 0.0
        _grad_norm_above_thresh_count = 0
        n_skipped += 1   # count as a corrective action
        print(f'[watchdog] Continuing training from step {step+2} with restored best weights.')

    run_ntp += accum_ntp
    run_vreg += accum_vreg
    n_run += 1
    steps_this_session += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg_ntp = run_ntp / n_run
        avg_vreg = run_vreg / n_run
        last_avg_ntp = avg_ntp
        run_ntp, run_vreg, n_run = 0.0, 0.0, 0
        elapsed = time.time() - t0
        gamma_val = model.gamma.item()
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        sec_per_step = elapsed / steps_this_session
        remaining = (TOTAL_STEPS - step - 1) * sec_per_step
        print(
            f'step {step+1:7d}/{TOTAL_STEPS}  '
            f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  lr={lr_now:.2e}  '
            f'grad={float(grad_norm):.2f}  gamma={gamma_val:.3f}  '
            f'alpha=[{alpha_str}]  '
            f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)'
        )
        log_f.write(json.dumps({
            'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'gamma': gamma_val, 'xi_alphas': alphas,
            'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
        }) + '\n')
        log_f.flush()

    if (step + 1) % EVAL_INTERVAL == 0:
        vl = evaluate()
        vppl = math.exp(vl)
        elapsed = time.time() - t0
        is_best = vppl < best_val_ppl
        best_marker = '  *** NEW BEST ***' if is_best else ''
        print(f'  >>> EVAL step {step+1:,}  val_loss={vl:.4f}  val_ppl={vppl:.2f}  '
              f'best={min(best_val_ppl, vppl):.2f}{best_marker}  ({elapsed:.0f}s)')
        log_f.write(json.dumps({
            'step': step + 1, 'val_loss': vl, 'val_ppl': vppl,
            'best_ppl': min(best_val_ppl, vppl),
        }) + '\n')
        log_f.flush()
        if is_best:
            best_val_ppl = vppl
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optim.state_dict(),
                'model_cfg': asdict(model_cfg),
                'v_theta_recipe': {'kind': V_THETA_KIND, 'K_mix': K_MIX, 'tau': TAU,
                                   'xi_channels': XI_CHANNELS},
                'step': step + 1, 'val_loss': vl, 'val_ppl': vppl,
                'gamma': model.gamma.item(), 'xi_alphas': model.xi_alpha_values(),
            }, _best_ckpt_path)
            print(f'      saved best -> {_best_ckpt_path.name}  (PPL {vppl:.2f})')

    if (step + 1) in ckpt_steps_set:
        vl = evaluate()
        save_checkpoint(step + 1, vl)
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

log_f.close()
total_elapsed = time.time() - t0
print(f'\nTraining complete ({total_elapsed:.0f}s / {total_elapsed/3600:.2f}h this session).')
print(f'Steps this session: {steps_this_session:,}  (skipped non-finite updates: {n_skipped})')
print(f'Best val PPL so far: {best_val_ppl:.2f}  ({_best_ckpt_path.name})')

In [ ]:
# ── Cell 7: Final evaluation + landscape diagnostics + summary ────
final_val = evaluate()
final_ppl = math.exp(final_val)
final_gamma = model.gamma.item()
final_alphas = model.xi_alpha_values()
total_elapsed = time.time() - t0

print(f'\n{"="*60}')
print(f'PHASE 4 FINAL  val_loss={final_val:.4f}  val_ppl={final_ppl:.2f}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}  params={n_params:,}')
print(f'  V_theta=SQ3(K_mix={K_MIX}, tau={TAU})  V_theta params={n_v_theta:,}')
print(f'  gamma={final_gamma:.4f}')
print(f'  alpha_final={final_alphas}')
print(f'  elapsed this session={total_elapsed:.0f}s ({total_elapsed/3600:.2f}h)')
print(f'{"="*60}')

save_checkpoint(TOTAL_STEPS, final_val, tag_suffix='_final')

# ── V_theta landscape diagnostics (structured-V_theta signature) ──
try:
    v_samples = []
    model.eval()
    for _ in range(10):
        xb, _ = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        with torch.enable_grad():
            h0 = model._embed(x)
            h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
            xis = model.xi_module(h_L.detach())
            h_for_v = model.ln_before_v(h_L) if model.ln_before_v is not None else h_L
            V_vals = model.V_theta(xis, h_for_v)
            v_samples.append(V_vals.detach().cpu().numpy().ravel())
    model.train()
    v_all = np.concatenate(v_samples)
    ls = {
        'mean': float(v_all.mean()), 'std': float(v_all.std()),
        'min': float(v_all.min()), 'max': float(v_all.max()),
        'range': float(v_all.max() - v_all.min()),
    }
    print('\nV_theta landscape stats:')
    for k, v in ls.items():
        print(f'  {k:6s}: {v:.4f}')
    with open(RESULTS_DIR / 'landscape_stats_phase4.json', 'w') as f:
        json.dump(ls, f, indent=2)
except Exception as e:
    print(f'Could not compute landscape stats: {e}')

# ── Loss curve from FULL training log (all sessions) ──
try:
    all_log = []
    with open(RESULTS_DIR / 'training_log.jsonl') as f:
        for line in f:
            e = json.loads(line)
            if 'val_ppl' in e:
                all_log.append(e)
    if all_log:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        steps_v = [e['step'] for e in all_log]
        ppls = [e['val_ppl'] for e in all_log]
        val_losses = [e['val_loss'] for e in all_log]

        ax1.plot(steps_v, val_losses, 'o-', color='darkorange', markersize=2, label='val loss')
        ax1.set_xlabel('step'); ax1.set_ylabel('loss (nats)')
        ax1.set_title('Validation loss (all sessions)')
        ax1.legend(); ax1.grid(True, alpha=0.3)

        ax2.plot(steps_v, ppls, 'o-', color='royalblue', markersize=2, label='val ppl')
        ax2.set_xlabel('step'); ax2.set_ylabel('perplexity')
        ax2.set_title('Validation perplexity (all sessions)')
        ax2.legend(); ax2.grid(True, alpha=0.3)

        fig.suptitle(
            f'Phase 4: Fock-PARFLM v2.1 + structured V_theta (SQ3 K={K_MIX}) OpenWebText\n'
            f'd={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}  params={n_params:,}',
            fontsize=11,
        )
        fig.tight_layout()
        fig_path = RESULTS_DIR / 'training_curve_phase4.png'
        fig.savefig(fig_path, dpi=150)
        print(f'Loss curve saved: {fig_path}')
        plt.show()
except Exception as e:
    print(f'Could not plot: {e}')

# ── Xi-alpha evolution ──
try:
    alpha_entries = []
    with open(RESULTS_DIR / 'training_log.jsonl') as f:
        for line in f:
            e = json.loads(line)
            if 'xi_alphas' in e:
                alpha_entries.append(e)
    if alpha_entries:
        K = len(alpha_entries[0]['xi_alphas'])
        fig2, ax = plt.subplots(figsize=(10, 5))
        for k in range(K):
            ax.plot([e['step'] for e in alpha_entries],
                    [e['xi_alphas'][k] for e in alpha_entries],
                    label=f'alpha_{k}')
        ax.set_xlabel('step'); ax.set_ylabel('alpha_k')
        ax.set_title('Xi-channel alpha evolution (all sessions)')
        ax.legend(); ax.grid(True, alpha=0.3)
        fig2.tight_layout()
        fig2_path = RESULTS_DIR / 'xi_alpha_evolution_phase4.png'
        fig2.savefig(fig2_path, dpi=150)
        print(f'Alpha plot saved: {fig2_path}')
        plt.show()
except Exception as e:
    print(f'Could not plot alpha evolution: {e}')

# ── Summary markdown ──
summary_path = RESULTS_DIR / 'training_summary.md'
with summary_path.open('w') as f:
    f.write('# Training summary — Phase 4 Fock-PARFLM v2.1 + structured V_theta (OpenWebText)\n\n')
    f.write('- model: FockMultiXiPARFLM v2.1 with SQ3 mixture-of-quadratic-wells V_theta\n')
    f.write(f'- corpus: OpenWebText (~{MAX_TRAIN_TOKENS//1_000_000}M train tokens)\n')
    f.write(f'- params: {n_params:,}  (V_theta SQ3: {n_v_theta:,})\n')
    f.write(f'- d={model_cfg.d}  L={model_cfg.L}  n_registers={model_cfg.n_registers}\n')
    f.write(f'- V_theta: SQ3 K_mix={K_MIX}  tau={TAU}  lambda_V={LAMBDA_V}\n')
    f.write(f'- xi_channels={model_cfg.xi_channels}  alpha_final={final_alphas}\n')
    f.write(f'- v_phi: structural_competitive  top_k={model_cfg.top_k}\n')
    f.write(f'- fixed_gamma={model_cfg.fixed_gamma}\n')
    f.write(f'- batch_size={BATCH_SIZE}  block_size={BLOCK_SIZE}  '
            f'grad_accum={GRAD_ACCUM}  steps={TOTAL_STEPS:,}\n')
    f.write(f'- seed={SEED}\n\n')
    f.write(f'Final val loss: {final_val:.6f} (ppl {final_ppl:.2f})\n')
    f.write(f'Final gamma: {final_gamma:.4f}\n')
    f.write(f'Final alpha: {final_alphas}\n\n')
    f.write('## Reference points\n\n')
    f.write('| Model | Corpus | PPL |\n')
    f.write('|-------|--------|-----|\n')
    f.write('| Fock-PARFLM v2.1 MLP V_theta (A2 ref) | TinyStories 16k | 8.95 |\n')
    f.write('| Fock-PARFLM v2.1 SQ3 K=8 V_theta (A2) | TinyStories 16k | 10.36 |\n')
    f.write(f'| Fock-PARFLM v2.1 SQ3 K={K_MIX} V_theta (this) | OpenWebText {TOTAL_STEPS//1000}k | **{final_ppl:.2f}** |\n')
print(f'Summary: {summary_path}')

In [ ]:
# ── Cell 8: (Optional) Push final checkpoint to HuggingFace Hub ───
PUSH_TO_HF = False
HF_REPO    = 'dimitarpg13/semsimula-fock-parflm-structured-vtheta-owt-phase4'
HF_TOKEN   = ''

if PUSH_TO_HF:
    from huggingface_hub import HfApi, create_repo
    token = HF_TOKEN or os.environ.get('HF_TOKEN', '')
    if not token:
        print('No HF token — skipping push. Set HF_TOKEN env var or paste above.')
    else:
        api = HfApi(token=token)
        try:
            create_repo(HF_REPO, repo_type='model', exist_ok=True, token=token)
        except Exception as e:
            print(f'  (repo may already exist: {e})')

        final_ckpt = CKPT_DIR / f'{CKPT_PREFIX}_step{TOTAL_STEPS}_final.pt'
        if final_ckpt.exists():
            print(f'Uploading {final_ckpt} to {HF_REPO}...')
            api.upload_file(
                path_or_fileobj=str(final_ckpt),
                path_in_repo=final_ckpt.name,
                repo_id=HF_REPO,
                token=token,
            )
            print('Upload complete.')
        else:
            print(f'Final checkpoint not found at {final_ckpt}')
else:
    print('HF push disabled. Set PUSH_TO_HF = True to upload.')